# 04 · Which diseases tend to follow which?

A different kind of network: here **each dot is a diagnosis**, not a patient. An arrow A → B means "people who get A are clearly more likely to get B within the next year" (at least 10 patients, plus a strict test because we try ~40,000 pairs).

Then we use **Girvan–Newman** from Lecture 3: keep cutting the most "bridge-like" link until the network falls apart, and draw the order it falls apart in as a family tree (dendrogram).

In [ ]:
import community as community_louvain
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from sklearn.metrics import adjusted_rand_score

from ehrgraph.eval.aggregate import write_aggregate
from ehrgraph.graphs.trajectory import build_population_trajectory_graph, symmetrize_max
from ehrgraph.netstats.dendrogram import girvan_newman_tree
from ehrgraph.project import setup
from ehrgraph.viz import dendrogram_figures as df
from ehrgraph.viz import network_figures as nf

p = setup()  # finds the data in data/omop/


nf.apply_style()

## First try: the raw billing codes

Built from the original ICD codes, the network falls into pieces. Let's see what those pieces are made of.

In [ ]:
raw = symmetrize_max(build_population_trajectory_graph(p.billing_codes)[0])
pieces = sorted(nx.connected_components(raw), key=len, reverse=True)
system = lambda code: code.split("/")[0]  # noqa: E731
print(f"{raw.number_of_nodes()} diagnoses, {raw.number_of_edges()} links")
for piece in pieces[:2]:
    print(f"piece of {len(piece)}:", dict(pd.Series([system(c) for c in piece]).value_counts()))
cross = sum(system(u) != system(v) for u, v in raw.edges())
print(f"links between ICD-9 and ICD-10 codes: {cross}")

In [ ]:
tree_raw = girvan_newman_tree(raw)
era = {"ICD9CM": df.CATEGORICAL[0], "ICD10CM": df.CATEGORICAL[1]}
fig, ax = plt.subplots(figsize=(14, 4.5))
colors = {c: era.get(system(c), df.OTHER) for c in raw}
df.plot_dendrogram(tree_raw, colors, labels=None, ax=ax, orientation="top")
ax.legend(
    handles=[
        plt.Rectangle((0, 0), 1, 1, color=era["ICD9CM"], label="ICD-9 (before Oct 2015)"),
        plt.Rectangle((0, 0), 1, 1, color=era["ICD10CM"], label="ICD-10 (after)"),
    ]
)
ax.set_title("Raw codes: the first split is just the 2015 code switch")
fig.tight_layout()
nf.save(fig, p.figures / "04_raw_codes_tree.png")
plt.show()

## Fixed: the standard concepts

Same rules, but each diagnosis is now its standard SNOMED concept (same in both eras).

In [ ]:
net = symmetrize_max(build_population_trajectory_graph(p.diagnoses)[0])
names = p.names(net.nodes)
pieces = sorted(nx.connected_components(net), key=len, reverse=True)
print(f"{net.number_of_nodes()} diagnoses, {net.number_of_edges()} links")
print(f"biggest piece: {len(pieces[0])} diagnoses")

tree = girvan_newman_tree(net)
best = tree.levels[tree.best_level]
louvain = community_louvain.best_partition(net, weight="weight", random_state=0)
groups = {n: i for i, g in enumerate(best) for n in g}
agree = adjusted_rand_score([groups[n] for n in net], [louvain[n] for n in net])
print(
    f"Girvan-Newman best cut: {len(best)} groups, modularity {tree.modularity[tree.best_level]:.3f}"
)
print(f"Louvain agrees with it: ARI {agree:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 15))
df.plot_dendrogram(tree, df.community_colors(tree), labels=names, ax=ax, orientation="left")
ax.legend(
    handles=df.community_legend(tree, names, net),
    title="group size: best-connected member",
    loc="upper left",
    bbox_to_anchor=(0.0, -0.04),
    ncol=2,
    fontsize=8,
    title_fontsize=8,
)
ax.set_title("Disease families (Girvan-Newman)")
fig.tight_layout()
nf.save(fig, p.figures / "04_disease_families.png", dpi=120)
plt.show()

In [ ]:
pd.DataFrame(
    [
        {
            "size": len(g),
            "diagnoses (most connected first)": "; ".join(
                names[n] for n in sorted(g, key=lambda n: -net.degree(n))[:6]
            ),
        }
        for g in sorted(best, key=len, reverse=True)
        if len(g) >= 3
    ]
)

In [ ]:
rows = []
for condition, g, t in (("raw_codes", raw, tree_raw), ("snomed_harmonized", net, tree)):
    biggest = max(nx.connected_components(g), key=len)
    for metric, value in {
        "n_nodes": g.number_of_nodes(),
        "n_edges": g.number_of_edges(),
        "largest_cc_size": len(biggest),
        "gn_best_n_communities": len(t.levels[t.best_level]),
        "gn_best_modularity": t.modularity[t.best_level],
        "cross_vocabulary_edges": sum(system(u) != system(v) for u, v in g.edges()),
    }.items():
        rows.append(
            dict(
                task="population",
                condition=condition,
                metric=metric,
                value=float(value),
                graph_variant="omop_top200_rr365d_support10_bonferroni_symmax",
            )
        )
rows.append(
    dict(
        task="population",
        condition="snomed_harmonized",
        metric="ari_gn_best_vs_louvain",
        value=float(agree),
        graph_variant="omop_top200_rr365d_support10_bonferroni_symmax",
    )
)
write_aggregate(rows, p.results / "trajectory_dendrogram.csv");

## What we learned

- **On the raw billing codes, the network is two separate worlds:** 69 ICD-10 codes and 47 ICD-9 codes, with **zero** links between them. The tree's first split is just the 2015 switch, not medicine. A nice example of a data artifact that looks like structure.
- **With standard concepts, that problem goes away:** one main piece of 60 diagnoses (79 in total, 142 links).
- **The families make clinical sense** (best cut: 13 groups, modularity 0.57):
  - lung and breathing problems (pulmonary congestion, respiratory failure, pleural effusion, pneumonia)
  - sepsis and kidney failure (septic shock, acute renal failure, bacteremia)
  - heart failure (congestive heart failure, cardiomyopathy, chronic pulmonary edema)
  - atrial fibrillation and flutter
  - pregnancy
  - type 2 diabetes complications
- **Louvain mostly agrees** (ARI 0.65): same big families, slightly different borders.
- **Caveat:** "A then B within a year" isn't cause and effect, and making the graph undirected throws away which one came first.